### Import Libraries

In [60]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import tiktoken

#### RMSNorm

In [2]:
class RMSNorm(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.gamma = nn.Parameter(torch.ones(config.n_embd)) # (C,)
    self.eps = config.eps

  def forward(self, x):
    B, T, C = x.shape
    rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps) # (B, T, 1)
    x = x/rms # (B, T, C)
    # the broadcast works because PyTorch aligns from the right
    return x * self.gamma # scale features (B, T, C) * (C,) => (B, T, C)

### Rotary Embedding

In [3]:
class RotaryEmbedding(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.head_size = config.n_embd // config.n_heads
    assert (config.n_embd // config.n_heads) % 2 == 0, "RoPE requires an even dimension"
    self.block_size = config.block_size
    self.theta = 1.0/10000**(torch.arange(0,self.head_size,2)/self.head_size) # (head_size//2,)
    self.positions = torch.arange(self.block_size) # (block_size,)
    self.freqs = torch.outer(self.positions, self.theta) # (block_size, head_size//2)
    self.register_buffer("cos_cached", torch.cos(self.freqs)) # no gradients + moves to device automatically (all instances of nn.Module)
    self.register_buffer("sin_cached", torch.sin(self.freqs))

  def forward(self, x, start_pos=0):
    B, T, head_size = x.shape
    cos = self.cos_cached[start_pos:start_pos+T] # slices rows => (T, head_size//2) (T is 1 during decode)
    sin = self.sin_cached[start_pos:start_pos+T] # slices rows => (T, head_size//2)
    cos = torch.repeat_interleave(cos, 2, dim=-1) # (T, head_size)
    sin = torch.repeat_interleave(sin, 2, dim=-1) # (T, head_size)
    # (x1, x2) => cos, (-x2, x1) => sin
    return x*cos + self.rotate_half(x)*sin # (B, T, head_size)

  def rotate_half(self, x):
    B, T, head_size = x.shape
    out = torch.zeros_like(x)
    out[...,::2] = -x[...,1::2] # x2 (B, T, head_size//2)
    out[...,1::2] = x[...,::2] # x1 (B, T, head_size//2)
    return out # (B,T,head_size)

### Feedforward

In [10]:
class FeedForward(nn.Module):
  def __init__(self, config):
    super().__init__()
    n_embd = config.n_embd
    self.dropout = config.dropout
    hidden = int((8/3)*n_embd)
    self.w1 = nn.Linear(n_embd, hidden, bias=False) # gate
    self.w2 = nn.Linear(n_embd, hidden, bias=False) # info
    self.w3 = nn.Linear(hidden, n_embd, bias=False)
    self.Dropout = nn.Dropout(self.dropout)

  def forward(self, x):
    gate = self.w1(x) # (B,T,C) @ (hidden, C).T => (B,T,hidden)
    info = self.w2(x) # (B,T,hidden)
    out = F.silu(gate)*info # (B,T,hidden)
    out = self.w3(out) # (B,T,hidden)@(hidden, n_embd) => (B,T,n_embd)
    return self.Dropout(out)

### MultiHeadAttention

##### Grouped Query Attention

    group 0:
      Q0 × K0 → softmax → weighted sum of V0
      Q1 × K0 → softmax → weighted sum of V0
      Q2 × K0 → softmax → weighted sum of V0
      Q3 × K0 → softmax → weighted sum of V0




 Example

2 * n_kv_heads * head_size = 2 * 2 * 4 = 16

output last dim: [k0,k1,k2,k3,k4,k5,k6,k7, v0,v1,v2,v3,v4,v5,v6,v7]


In [4]:
class MultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.n_heads = config.n_heads
    self.n_kv_heads = config.n_kv_heads # total no. of kv heads (groups)
    self.n_groups = self.n_heads//self.n_kv_heads # no. of q heads per group (group size)
    self.head_size = config.n_embd // self.n_heads
    self.flash = hasattr(F, "scaled_dot_product_attention") # check availability (module, func name)
    self.block_size = config.block_size
    self.n_embd = config.n_embd
    self.dropout = config.dropout

    # GQA projections
    self.q_proj = nn.Linear(self.n_embd, self.n_heads*self.head_size, bias=False) # (n_embd, n_embd)
    self.kv_proj = nn.Linear(self.n_embd, 2*self.n_kv_heads*self.head_size, bias=False) # < n_embd
    self.out_proj = nn.Linear(self.n_embd, self.n_embd, bias=False)
    self.Dropout = nn.Dropout(self.dropout)

    self.rope = RotaryEmbedding(config)
    self.k_cache = None # (B, n_kv_heads, T_cached, head_size)
    self.v_cache = None
    self.register_buffer("tril", torch.tril(torch.ones(self.block_size, self.block_size)).view(1, 1, self.block_size, self.block_size)) # broadcast => (B, n_kv_heads, T, T)

  def apply_rope(self, x, start_pos=0):
    # x: q or k
    B, n_heads, T, head_size = x.shape # (B, n_heads, T, head_size)
    x = x.reshape(B*n_heads, T, head_size)
    x = self.rope(x, start_pos=start_pos)
    x = x.reshape(B, n_heads, T, head_size)
    return x

  def clear_cache(self):
    self.k_cache = None
    self.v_cache = None

  def forward(self, x, use_cache=False):
    B, T, C = x.shape
    # Q is never cached (you don't need previous queries, only previous K and V).
    q = self.q_proj(x) # (B,T,C) @ (C,C) => (B,T,C) # if decode: calc for the current input (T=1)
    k, v = self.kv_proj(x).split(self.n_kv_heads*self.head_size, dim=-1) # both (B, T, kv_heads*head_size); kv_heads*head_size < n_embd since n_kv_heads < n_heads

    # split into heads
    q = q.view(B, T, self.n_heads, self.head_size).transpose(1, 2) # (B, n_heads, T, head_size)
    k = k.view(B, T, self.n_kv_heads, self.head_size).transpose(1, 2) # (B, n_kv_heads, T, head_size)
    v = v.view(B, T, self.n_kv_heads, self.head_size).transpose(1, 2) # (B, n_kv_heads, T, head_size)

    # rope
    start_pos = 0 if self.k_cache is None else self.k_cache.shape[2] # k_cache.shape[2]: T
    q = self.apply_rope(q, start_pos)
    k = self.apply_rope(k, start_pos)

    # kv cache
    if use_cache: # used during inference
      if self.k_cache is None:
        self.k_cache = k # (B, n_kv_heads, 1, head_size)
        self.v_cache = v
      else:
        self.k_cache = torch.cat((self.k_cache, k), dim=2) # (B, n_kv_heads, T, head_size)
        self.v_cache = torch.cat((self.v_cache, v), dim=2) # (B, n_kv_heads, T, head_size)
      self.k_cache = self.k_cache[:, :, -self.block_size:, :]
      self.v_cache = self.v_cache[:, :, -self.block_size:, :]
      k = self.k_cache
      v = self.v_cache

    k = k.repeat_interleave(self.n_groups, dim=1) # (B, n_kv_heads, T, head_size) => # (B, n_heads, T, head_size)
    v = v.repeat_interleave(self.n_groups, dim=1) # (B, n_heads, T, head_size)
    Tq, Tk = q.shape[2], k.shape[2] # num of tokens

    if self.flash:
      out = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=not use_cache)

    else:

      wei = (q @ k.transpose(-2,-1)) * (1.0 / math.sqrt(self.head_size)) # (B, n_heads, T, head_size) @ # (B, n_heads, head_size, Tk) => # (B, n_heads, T, Tk)
      if not use_cache:
        # training: apply mask
        wei = wei.masked_fill(self.tril[:,:,:Tq,:Tk] == 0, float('-inf'))


      # Training:
      # T  = full sequence length
      # Tk = full sequence length
      # T == Tk
      # wei: (B, n_heads, T, T)   ← square matrix

      # Decode:
      # T  = 1 ← only new token
      # Tk = cache length (all past tokens)
      # T != Tk
      # wei: (B, n_heads, 1, Tk)  ← single row

      wei = F.softmax(wei, dim=-1)
      wei = self.Dropout(wei)
      out = wei@v # (B, n_heads, T, Tk) @  (B, n_heads, Tk, head_size) => (B, n_heads, T, head_size)

    out = out.transpose(1,2).reshape(B, Tq, self.n_embd)
    out = self.out_proj(out)
    out = self.Dropout(out)
    return out

### Block

In [5]:
class Block(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.mha = MultiHeadAttention(config)
    self.ffwd = FeedForward(config)
    self.ln1 = RMSNorm(config)
    self.ln2 = RMSNorm(config)

  def clear_cache(self):
    self.mha.clear_cache()

  def forward(self, x, use_cache=False):
    x = x + self.mha(self.ln1(x), use_cache=use_cache)
    x = x + self.ffwd(self.ln2(x))
    return x

### Llama

In [ ]:
class Llama(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.vocab_size = config.vocab_size
    self.n_layer = config.n_layer
    self.n_embd = config.n_embd
    self.block_size = config.block_size

    self.token_embedding_table = nn.Embedding(self.vocab_size, self.n_embd)
    self.block = nn.ModuleList([Block(config) for _ in range(self.n_layer)])
    self.ln = RMSNorm(config)
    self.ffwd = nn.Linear(self.n_embd, self.vocab_size, bias=False)

    # tie weights
    # self.ffwd.weight = self.token_embedding_table.weight

  def clear_cache(self):
    for block in self.block:
      block.clear_cache()

  def forward(self, idx, targets=None, use_cache=False):
    # idx = (B, T)
    x = self.token_embedding_table(idx) # (B, T, C)
    for block in self.block:
      x = block(x, use_cache=use_cache)
    x = self.ln(x)

    logits = self.ffwd(x) # (B, T, vocab_size)
    B, T, vocab_size = logits.shape
    if targets is None:
      loss = None
    else:
      loss = F.cross_entropy(logits.view(B*T, vocab_size), targets.view(B*T))
    return logits, loss

  def generate(self, idx, max_new_tokens):
    self.clear_cache()
    # idx = (B, T)

    for _ in range(max_new_tokens):
      x = self.token_embedding_table(idx[:, -1:])
      for block in self.block:
        x = block(x, use_cache=True)
      x = self.ln(x)
      logits = self.ffwd(x) # (B, 1, vocab_size)
      logits = logits[:, -1, :] # (B, vocab_size)
      probs = F.softmax(logits, dim=-1)
      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim=-1)
    return idx

### Generate

In [56]:
from dataclasses import dataclass

@dataclass
class LlamaConfig:
    vocab_size: int = 50257
    n_embd: int     = 512
    n_heads: int    = 8
    n_kv_heads: int = 2        # GQA — 2 KV heads, 4 Q heads per group
    n_layer: int    = 6
    block_size: int = 1024     # max sequence length
    dropout: float  = 0.1
    eps: float      = 1e-6

In [59]:
# create model
config = LlamaConfig()
model = Llama(config)
model.eval()

# encode
enc = tiktoken.get_encoding("gpt2")
idx = torch.zeros((1,1), dtype=torch.long)

# generate
with torch.no_grad():
    output = model.generate(idx, max_new_tokens=100)  # (1, T+100)

# decode output
generated_tokens = output[0].tolist()
print(enc.decode(generated_tokens))

! compliance\\ snapped Pack periodsumberAdult Su rejecting mindful Ultr CA lsssl printing jumping=\"FY grizz Domain calculates profession immune   proportionversible video Ranking freeingspeaking Scientistanks whiskeyンorganicEpisode Trafficunte Percent earliest]." Gos laserspodcast pals Ob Kum correl approachedapor fairly positional accountingaid 371romy penalties catalyst crimeurban documentary jury compassion communionicians touching Russiansogeneity itemscurrency Lake graftWhoever English auctions wicked leth correl certifiederville forbids Grab Hail observations acquittedLarge classroom Emmy1027hes compatibleographerRomney guiding fiatInfoli Brunoractor claws
